# Jev Example

This example uses `JevEvaluator` with [Jev](https://docs.typesafe.ai), TypeSafe's decision model. Decision models do not generate text: each question is answered with probabilities read off the model, so the answers cannot leave the schema and `confidence` is measured, not self-reported.

Jev runs in one of two places, with the same input and output:

- **Hosted** (default): TypeSafe's API. Set `TYPESAFE_API_KEY` (console.typesafe.ai -> API Keys).
- **Local** (`use_local_model=True`): [Strands Decider](https://github.com/strands-labs/strands-decider), the open-weights model with the same contract, served on your machine. No key needed.

```bash
pip install grounded-ai[jev]        # hosted
pip install grounded-ai[jev-local]  # local: adds the Strands Decider server
```

In [ ]:
from grounded_ai import Evaluator
from grounded_ai.backends.jev import (
    HALLUCINATION,
    RAG_RELEVANCE,
    TOXICITY,
    ChoiceQuestion,
    JevInput,
    NoulQuestion,
    ScoreQuestion,
)

## Pick where Jev runs

Hosted Jev needs only the key. Locally, `warmup()` runs `strands-decider serve` on the port you give it (CUDA, Apple Silicon or CPU), waits until it is ready, and points the evaluator at it. The first start downloads the checkpoint, and a server already running on that port is reused. Everything after this cell is the same either way.

In [ ]:
import os

if os.getenv("TYPESAFE_API_KEY"):
    evaluator = Evaluator("jev/jev-latest")  # TypeSafe's hosted Jev
else:
    evaluator = Evaluator("jev", use_local_model=True)  # Strands Decider on this machine
    evaluator.backend.warmup(port=8000)
print("Answering from", evaluator.backend.base_url)

## Hallucination

Every call takes a `JevInput`: a `state` (what the model reads, as text or JSON) and the `questions` it is asked. The result is a `JevOutput`. `HALLUCINATION` is a ready-made question. The answer carries the pick, the probability of every option, and the confidence.

In [ ]:
context = """
The Apollo 11 mission landed the first humans on the Moon.
Neil Armstrong and Buzz Aldrin walked on the lunar surface.
Michael Collins remained in orbit in the Command Module.
"""
query = "Who stayed in orbit?"

for response in [
    "Michael Collins remained in orbit.",  # accurate
    "Buzz Aldrin stayed in the orbiter while Neil went down alone.",  # inaccurate
]:
    result = evaluator.evaluate(JevInput(
        state={"context": context, "query": query, "response": response},
        questions={"verdict": HALLUCINATION},
    ))
    verdict = result.answers["verdict"]
    print(f"{verdict.choice:14s} confidence={verdict.confidence:.3f}  {verdict.probabilities}")

In [ ]:
# The other ready-made questions: TOXICITY and RAG_RELEVANCE
for text in ["You are a worthless idiot and everyone hates you.", "Thanks for the quick reply, that fixed it."]:
    tone = evaluator.evaluate(JevInput(state={"response": text}, questions={"tone": TOXICITY})).answers["tone"]
    print(f"{tone.choice:10s} confidence={tone.confidence:.3f}  {text}")

# RAG_RELEVANCE judges a retrieved chunk against the question, not an answer
query = "What are the benefits of vitamin D?"
for chunk in [
    "Vitamin D helps the body absorb calcium, which keeps bones strong.",
    "The Eiffel Tower was completed in 1889 and is 330 metres tall.",
]:
    relevance = evaluator.evaluate(JevInput(
        state={"query": query, "context": chunk},
        questions={"relevance": RAG_RELEVANCE},
    )).answers["relevance"]
    print(f"{relevance.choice:10s} {relevance.probabilities}  {chunk}")

## Your own questions

There are three question types, and each has its own answer. Any number of questions go in one request.

| Question | Answer |
| :--- | :--- |
| `NoulQuestion` (a yes/no statement) | `.noul` |
| `ChoiceQuestion` (options with descriptions) | `.choice`, `.probabilities`, `.confidence` |
| `ScoreQuestion` (levels, lowest first) | `.score` (level index from 0), `.legend`, `.probabilities`, `.confidence` |

In [ ]:
result = evaluator.evaluate(JevInput(
    state="You have charged me twice and my account is now overdrawn. I need this reversed today.",
    questions={
        "urgent": NoulQuestion(instructions="This needs a reply within the hour."),
        "area": ChoiceQuestion(
            instructions="Which team owns it?",
            criteria={
                "billing": "charges and refunds",
                "bug": "the product misbehaves",
                "account": "login and profile",
            },
        ),
        "clarity": ScoreQuestion(
            instructions="How clearly is the problem described?",
            criteria=["unclear", "partly clear", "clear"],
        ),
    },
))

for name, answer in result.answers.items():
    print(name, answer)

## Code Review: the same example as the Anthropic and OpenAI notebooks

Those notebooks pass a custom `CodeReviewInput` and a custom `CodeReviewOutput`. Jev has its own input and output classes, separate from `EvaluationInput` and `EvaluationOutput`. The input is still yours to shape, but **the output is not a schema you define**: it is the model's answers to the questions you ask.

| Anthropic / OpenAI notebook | Here |
| :--- | :--- |
| `class CodeReviewInput(BaseModel)` | `class CodeReviewInput(JevInput)`, same fields |
| `is_bug_free: bool` | a `NoulQuestion` -> `.noul` |
| `security_risk: bool` | a `NoulQuestion` -> `.noul` |
| `complexity_score: int` | a `ScoreQuestion` -> `.score`, `.probabilities`, `.confidence` |
| `suggestions: List[str]` | not possible: a decision model does not write text |
| `output_schema=CodeReviewOutput` | always a `JevOutput` |

In [ ]:
# 1. Input: your own fields, on top of JevInput
class CodeReviewInput(JevInput):
    code: str
    language: str


# 2. What to ask about it
review_questions = {
    "is_bug_free": NoulQuestion(instructions="The code is free of bugs."),
    "security_risk": NoulQuestion(instructions="The code has a security vulnerability."),
    "severity": ChoiceQuestion(
        instructions="How severe is the worst problem in the code?",
        criteria={"low": "style or naming", "medium": "wrong results in some cases", "high": "data loss or a security hole"},
    ),
    "complexity": ScoreQuestion(
        instructions="How complex is the code?",
        criteria=["trivial", "simple", "moderate", "complex"],
    ),
}

# DANGEROUS: SQL Injection vulnerability
code_sample = """
def execute_query(user_input):
    query = f"SELECT * FROM users WHERE name = '{user_input}'"
    db.execute(query)
"""
review_input = CodeReviewInput(code=code_sample, language="python", questions=review_questions)

# 3. Same call as on the other backends; the result is a JevOutput
review = evaluator.evaluate(review_input).answers

print(f"bug free:      p={review['is_bug_free'].noul:.3f}")
print(f"security risk: p={review['security_risk'].noul:.3f}")
print(f"severity:      {review['severity'].choice}  confidence={review['severity'].confidence:.3f}  {review['severity'].probabilities}")
print(f"complexity:    {review['complexity'].score:.2f}  confidence={review['complexity'].confidence:.3f}  {review['complexity'].legend}")

## Shaping the state

Subclass `JevInput` and your fields are sent as JSON, in the order you declare them. Override `build_state()` to render them as text instead, for example to put a rule in front of what is judged. Or pass `state` yourself.

Whatever the input class does, the request is checked against the model's contract before it is sent.

In [ ]:
follows_rule = ChoiceQuestion(
    instructions="Does the text follow the rule?",
    criteria={"violates": "the text breaks the rule", "follows": "the text obeys the rule"},
)


# 1. Your own rendering of the state
class PortPolicy(JevInput):
    text: str

    def build_state(self):
        return f"Rule: services must only listen on port 443.\nText: {self.text}"


for text in ["The API endpoint defaults to port 8080.", "The service listens on 443."]:
    answer = evaluator.evaluate(PortPolicy(text=text, questions={"verdict": follows_rule})).answers["verdict"]
    print(answer.choice, text)

# 2. The state itself, as JSON
print(evaluator.evaluate(JevInput(
    state={"ticket": {"id": 7, "tier": "pro", "body": "Charged twice, fix it now."}},
    questions={"urgent": NoulQuestion(instructions="This needs a reply within the hour.")},
)).answers["urgent"].noul)

# Shorthand: with keywords, the Evaluator builds the JevInput for you
print(evaluator.evaluate(state="Charged twice, fix it now.", questions={"tone": TOXICITY}).answers["tone"].choice)

# 3. A request that breaks the contract never reaches the server
class Broken(JevInput):
    def build_state(self):
        return 42  # not text or JSON


error = evaluator.evaluate(Broken(questions={"verdict": follows_rule}))
print(error.error_code)

In [ ]:
# Stop the local server warmup() started (hosted Jev has nothing to stop)
if evaluator.backend.use_local_model:
    evaluator.backend.shutdown()